In [1]:
!pip install datasets

In [2]:
from datasets import load_dataset, Dataset
import re
from google.colab import drive

In [3]:
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
dataset_id = 'aitamilnadu/tamil_stories'
split_name = 'train'
num_records = 10000

In [5]:
streaming_dataset = load_dataset(dataset_id, split=split_name, streaming=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

In [6]:
first_n_records_iterable = streaming_dataset.take(num_records)

In [7]:
first_n_records_dataset = Dataset.from_list(list(first_n_records_iterable))

In [8]:
first_n_records_dataset

Dataset({
    features: ['template_id', 'template_lang', 'inputs', 'targets'],
    num_rows: 1202
})

In [9]:
first_n_records_dataset[0]

{'template_id': 1,
 'template_lang': "['tam']",
 'inputs': "கீழே கொடுக்கப்பட்டுள்ள கதைக்குப் பொருத்தமான தலைப்பைக் கொடு.\nகதை: அன்றைக்குக் கடைசி ஆடி. ஊர் முழுக்க தோசை வாசனை கம்ம்மென்று முறுகல் மணல். ஆட்டுரல்களில் சட்னி ஆட்டுகிற கடகடா சப்தம். வழக்கத்துக்கு மாறாக... காலில் சக்கரம் கட்டிக் கொண்டு பறக்கிறான் சீனிவாசன். ஒரே பரபரப்பு. அங்கேயும் இங்கேயுமாய்ப் பாய்கிறான். ஆளைக் கையில் பிடிக்க முடியவில்லை. மூக்கின் மேல் நின்ற கோபம் நாலா பக்கமும் சிதறித் தெறிக்கிறது. இன்னார் மீது என்று கணக்கில்லை. சகட்டு மேனிக்கு ''சள் சள்'' ளென்று சீறுகிறான். மண்ணடிக்க டக்கர் போயிருக்கிறது. வேலை செய்யாமல் கூலியாட்கள் ஏய்த்து விடுவார்களே என்கிற பதற்றம், அவனுக்குள். சம்பளம் வாங்கத் தீயாய் வருகிற ஆட்கள். பாடுபடாமல் தேங்கின தண்ணீராகத் தேய்ந்து போகிற வஞ்சகம். ச்சே! நினைத்தாலே மனசு கிடந்து கொதிக்கிறது. என்றைக்குமில்லாத அதிசயமாக விடிவதற்கு முன்பே விழித்துவிட்டான் சீனிவாசன். டீக்கடை போய், ஓடைக்குப் போய், பல்லையும் தேய்த்துவிட்டு - ''என்ன ரெடியா?'' என்று காலில் கொதி நீரை ஊற்றிக் கொண்டு நிற்கிறான். வேணித்தாய்க்கு எரிச்சலா

In [10]:
from tqdm import tqdm

In [11]:
text = ''
for i in tqdm(range(1202)):
  temp_text = first_n_records_dataset[i]['inputs']
  text = text + '\n\n' + temp_text

100%|██████████| 1202/1202 [00:01<00:00, 916.24it/s]


In [12]:
len(text.split())

312429

In [13]:
def remove_non_allowed_content(text: str) -> str:
    # The pattern matches any character that is NOT one of the following:
    # 1. \u0D80-\u0DFF: Sinhala characters (Unicode block)
    # 2. a-zA-Z: English/Latin letters (both cases)
    # 3. 0-9: Numbers
    # 4. \s: Whitespace characters (space, tab, newline)
    # 5. \.,!?:;'"(){}\[\]@#$&*%+-=/\\<>=|_~^: Common punctuation and symbols

    # We use re.UNICODE flag to ensure correct handling of Unicode characters like Sinhala.
    # The character class [^...] means "match any character that is NOT inside this class."
    pattern = r'[^\u0B80-\u0BFF0-9\s\.,!?:;\'"(){}\[\]@#$&*%+-=/\\<>=|_~^]'

    # Replace any matched characters (the unwanted ones) with an empty string
    cleaned_text = re.sub(pattern, '', text, flags=re.UNICODE)

    return cleaned_text

In [14]:
cleaned_output = remove_non_allowed_content(text)

In [15]:
len(cleaned_output.split())

311804

In [16]:
drive_path = "/content/drive/MyDrive/ICTer_Workshop/tamil_phrases_filtered.txt"

In [17]:
with open(drive_path, 'w') as f:
    f.write(cleaned_output)